# Module 1 · Fondamentaux du NLP — Travaux pratiques
**Togo AI Lab · Formation IA · Version IT** · Durée : environ 1 h 30

**Ce que vous allez faire :** apprendre à une machine à reconnaître le *sujet* d'un message d'usager (santé, état civil ou éducation), en partant de zéro : nettoyer le texte, le découper, le compter, puis entraîner un petit classifieur.

**À la fin, vous saurez :**
1. nettoyer et découper un texte (normalisation, tokenisation) ;
2. transformer des textes en nombres (sac de mots, TF-IDF) ;
3. entraîner et évaluer un classifieur de textes ;
4. expliquer pourquoi la machine se trompe parfois.

**Comment utiliser ce notebook (Google Colab)**
- Une cellule de code s'exécute avec **Maj + Entrée**. Exécutez-les **dans l'ordre**, de haut en bas.
- Les cellules marquées **👉 À vous** contiennent des `____` à remplacer par votre code. Lisez la consigne juste au-dessus.
- Si vous êtes bloqué, relisez le message d'erreur : il indique presque toujours la ligne en cause.
- Toutes les données sont **fictives** : aucun message ne provient d'une vraie personne.

## 0 · Préparation
Exécutez cette cellule : elle charge les outils (bibliothèques) dont nous avons besoin.

In [ ]:
import re, unicodedata, random
from collections import Counter

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

pd.set_option("display.max_colwidth", 90)
random.seed(42)
print("Outils prêts ✔")

## 1 · Découvrir les données
Notre *corpus* : 72 messages fictifs adressés à des services publics, classés en 3 catégories. **Exécutez la cellule suivante sans la lire en détail** : elle contient simplement les données.

In [ ]:
# Données fictives (messages inventés pour la formation)
CORPUS = [
    ('Mon enfant a de la fièvre depuis deux jours, où trouver un médecin ?', 'sante'),
    ('Je cherche le centre de santé le plus proche pour une consultation.', 'sante'),
    ("Comment obtenir un rendez-vous à l'hôpital pour mon père ?", 'sante'),
    ('La pharmacie de garde est-elle ouverte ce dimanche ?', 'sante'),
    ('Ma mère doit faire une analyse de sang, quels documents apporter ?', 'sante'),
    ('Quand a lieu la prochaine campagne de vaccination dans le quartier ?', 'sante'),
    ("J'ai mal au ventre depuis ce matin, que dois-je faire ?", 'sante'),
    ('Le dispensaire manque de médicaments pour le paludisme.', 'sante'),
    ('Je voudrais savoir où faire vacciner mon bébé.', 'sante'),
    ("L'infirmière du centre de santé m'a demandé une ordonnance.", 'sante'),
    ('Où passer une radio de la poitrine à Lomé ?', 'sante'),
    ("Ma fille est enceinte, elle a besoin d'une consultation prénatale.", 'sante'),
    ("Quels sont les horaires de consultation à l'hôpital régional ?", 'sante'),
    ("Mon voisin est tombé malade, faut-il appeler l'ambulance ?", 'sante'),
    ('Je dois renouveler mon traitement, le médecin est-il disponible ?', 'sante'),
    ('Y a-t-il un médecin de garde au dispensaire cette nuit ?', 'sante'),
    ('Le vaccin contre la rougeole est-il gratuit pour les enfants ?', 'sante'),
    ("Mon grand-père a une forte toux, il a besoin d'un docteur.", 'sante'),
    ("Je cherche une maternité proche de chez moi pour l'accouchement.", 'sante'),
    ('Comment obtenir un carnet de santé pour mon enfant ?', 'sante'),
    ('Le centre de santé est fermé, où aller pour soigner une blessure ?', 'sante'),
    ('Quels médicaments sont remboursés pour le traitement du diabète ?', 'sante'),
    ("Ma sœur a besoin d'un certificat médical après sa consultation.", 'sante'),
    ("Où acheter des médicaments moins chers près de l'hôpital ?", 'sante'),
    ("Comment obtenir un extrait d'acte de naissance pour mon fils ?", 'etat_civil'),
    ('Je dois déclarer la naissance de ma fille à la mairie.', 'etat_civil'),
    ("Quels documents faut-il pour établir une carte d'identité nationale ?", 'etat_civil'),
    ('Mon acte de mariage est perdu, comment en demander une copie ?', 'etat_civil'),
    ('Je veux renouveler mon passeport, quelles sont les étapes ?', 'etat_civil'),
    ('Où déposer une demande de certificat de nationalité ?', 'etat_civil'),
    ('Combien coûte un acte de décès et quel est le délai ?', 'etat_civil'),
    ('Le nom de mon père est mal écrit sur mon acte de naissance.', 'etat_civil'),
    ("Je cherche l'adresse du bureau d'état civil de ma commune.", 'etat_civil'),
    ('Quels papiers pour célébrer un mariage à la mairie ?', 'etat_civil'),
    ('Mon passeport expire bientôt, je dois faire une nouvelle demande.', 'etat_civil'),
    ("Comment faire une demande de jugement supplétif d'acte de naissance ?", 'etat_civil'),
    ("Ma carte d'identité a été volée, que faire ?", 'etat_civil'),
    ('Je voudrais légaliser une copie de mon diplôme à la mairie.', 'etat_civil'),
    ('Quel est le délai pour recevoir un certificat de résidence ?', 'etat_civil'),
    ('Il me faut un extrait de casier judiciaire pour un dossier.', 'etat_civil'),
    ("Mon mari est décédé, comment obtenir l'acte de décès ?", 'etat_civil'),
    ('La mairie a perdu mon dossier de changement de nom.', 'etat_civil'),
    ('Où retirer mon certificat de nationalité déjà demandé ?', 'etat_civil'),
    ("Je n'ai pas d'acte de naissance, comment régulariser ma situation ?", 'etat_civil'),
    ('Quelles pièces fournir pour une demande de passeport pour mon enfant ?', 'etat_civil'),
    ("Le guichet de l'état civil ferme à quelle heure ?", 'etat_civil'),
    ('Je veux faire modifier ma date de naissance sur mon acte.', 'etat_civil'),
    ('Comment faire une procuration pour retirer un acte à la mairie ?', 'etat_civil'),
    ('Comment inscrire mon fils en sixième dans un collège public ?', 'education'),
    ('Quand commence la rentrée scolaire cette année ?', 'education'),
    ("Mon enfant a besoin d'un certificat de scolarité pour son dossier.", 'education'),
    ("Quels sont les frais d'inscription à l'université de Lomé ?", 'education'),
    ('Où retirer le relevé de notes du baccalauréat de ma fille ?', 'education'),
    ("L'école de mon village manque d'enseignants en mathématiques.", 'education'),
    ("Comment obtenir une bourse d'études pour mon fils à l'université ?", 'education'),
    ("Quand sont publiés les résultats de l'examen du BEPC ?", 'education'),
    ('Je cherche une école primaire proche de chez moi pour mon enfant.', 'education'),
    ('Le professeur de mon fils est souvent absent en classe.', 'education'),
    ("Quels documents pour l'inscription en première année de licence ?", 'education'),
    ('Ma fille a réussi son examen, où retirer son diplôme ?', 'education'),
    ("Comment s'inscrire au concours d'entrée à l'école d'infirmiers ?", 'education'),
    ("Les élèves de l'école n'ont pas de manuels scolaires.", 'education'),
    ('Je voudrais transférer mon enfant dans un autre collège.', 'education'),
    ('Quelle est la date limite pour déposer le dossier de bourse ?', 'education'),
    ('Le lycée organise-t-il des cours de rattrapage pour les élèves ?', 'education'),
    ("Mon fils passe le baccalauréat cette année, comment s'inscrire ?", 'education'),
    ("Où trouver le calendrier des examens de fin d'année scolaire ?", 'education'),
    ("La directrice de l'école refuse d'inscrire mon enfant en classe.", 'education'),
    ('Je cherche une formation professionnelle pour mon neveu après le collège.', 'education'),
    ('Comment obtenir une attestation de réussite pour mon fils ?', 'education'),
    ('Les frais de scolarité du lycée ont-ils augmenté cette année ?', 'education'),
    ('Quel est le programme des cours en classe de troisième ?', 'education'),
]

df = pd.DataFrame(CORPUS, columns=["message", "categorie"])
print(len(df), "messages chargés")

In [ ]:
df.sample(5, random_state=1)

In [ ]:
df["categorie"].value_counts()

**👉 À vous — Exercice 1.** Affichez 3 messages de la catégorie `education`.  
*Indice : on filtre un tableau avec `df[df["categorie"] == "..."]`, puis `.head(3)`.*

In [ ]:
df[df["categorie"] == "education"].head(3)

## 2 · Découper le texte : la tokenisation
Une machine ne « lit » pas des phrases : elle travaille sur des **morceaux** appelés *tokens*. Le plus simple : couper aux espaces.

In [ ]:
phrase = "L'infirmière du centre de santé m'a demandé une ordonnance."
print(phrase.split())

Remarquez `"L'infirmière"` et `"m'a"` : l'apostrophe colle des mots ensemble, et le point reste attaché à `"ordonnance."`.  
Avec une *expression régulière*, on découpe mieux : on garde uniquement les suites de lettres.

In [ ]:
def decouper(texte):
    return re.findall(r"\w+", texte, flags=re.UNICODE)

print(decouper(phrase))

**👉 À vous — Exercice 2.** Combien de tokens contient le **premier message** du tableau ?  
*Indice : `df["message"].iloc[0]` donne le premier message, `len(...)` compte les éléments d'une liste.*

In [ ]:
premier = df["message"].iloc[0]
print(premier)
print(len(decouper(premier)), "tokens")

## 3 · Nettoyer : normaliser le texte
Deux textes qui *semblent* identiques pour nous peuvent être **différents pour la machine**. Exemple :

In [ ]:
a = "é"                      # un seul caractère
b = "e\u0301"                # « e » + accent collé : s'affiche pareil !
print(a, b, "→ égaux ?", a == b, "| longueurs :", len(a), len(b))
print("après normalisation NFC → égaux ?", unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC", b))

> **Pourquoi c'est important pour nous :** plusieurs langues togolaises s'écrivent avec des lettres spéciales (ɖ, ɛ, ɔ, ŋ…). Selon le clavier ou le logiciel, elles peuvent être encodées de deux façons. Sans normalisation, la machine croit voir deux mots différents.

Construisons une fonction de nettoyage : normaliser Unicode, mettre en minuscules, découper, retirer les « mots vides » (très fréquents mais peu informatifs).

In [ ]:
MOTS_VIDES = {"le", "la", "les", "un", "une", "des", "de", "du", "d", "l", "à", "a", "au", "aux", "et", "en",
              "je", "j", "mon", "ma", "mes", "ce", "ces", "se", "que", "qui", "quel", "quels", "quelle",
              "est", "il", "elle", "on", "y", "pour", "pas", "sur", "dans", "par", "ou", "où", "m", "n", "s", "c",
              # verbes et mots « passe-partout » des demandes
              "comment", "faire", "obtenir", "voudrais", "cherche", "savoir", "sont", "veux", "dois", "besoin"}

def nettoyer(texte, retirer_mots_vides=True):
    texte = unicodedata.normalize("NFC", texte).lower()
    mots = decouper(texte)
    if retirer_mots_vides:
        mots = [m for m in mots if m not in MOTS_VIDES]
    return mots

print(nettoyer("Mon enfant a de la fièvre depuis deux jours, où trouver un médecin ?"))

**👉 À vous — Exercice 3.** Appliquez `nettoyer` à la phrase `"Où retirer mon certificat de nationalité déjà demandé ?"` et observez quels mots ont disparu.

In [ ]:
nettoyer("Où retirer mon certificat de nationalité déjà demandé ?")

## 4 · Compter les mots : quels mots reviennent le plus ?

In [ ]:
df["mots"] = df["message"].apply(nettoyer)
tous = Counter(m for liste in df["mots"] for m in liste)
tous.most_common(10)

**👉 À vous — Exercice 4.** Trouvez les **5 mots les plus fréquents de la catégorie `sante`**. Ces mots vous semblent-ils bien décrire le sujet ?  
*Indice : on filtre d'abord les lignes `sante`, puis on compte comme ci-dessus.*

In [ ]:
sante = df[df["categorie"] == "sante"]
compteur = Counter(m for liste in sante["mots"] for m in liste)
compteur.most_common(5)

## 5 · Transformer le texte en nombres : le « sac de mots »
Pour qu'un algorithme travaille, chaque message devient une **ligne de nombres** : combien de fois chaque mot du vocabulaire apparaît. L'ordre des mots est ignoré, d'où le nom de « sac de mots ».

In [ ]:
exemples = ["mon enfant a de la fièvre, où trouver un médecin ?",
            "comment obtenir un acte de naissance pour mon enfant ?",
            "quand commence la rentrée scolaire pour mon enfant ?"]
cv = CountVectorizer(tokenizer=nettoyer, lowercase=False, token_pattern=None)
matrice = cv.fit_transform(exemples)
pd.DataFrame(matrice.toarray(), columns=cv.get_feature_names_out(), index=["msg 1", "msg 2", "msg 3"])

Lecture : chaque colonne est un mot, chaque ligne un message. Le mot `enfant` vaut 1 dans les trois messages : il n'aide donc pas à les distinguer. Les mots `médecin`, `acte`, `rentrée` sont, eux, **caractéristiques**.

**👉 À vous — Exercice 5.** Construisez le sac de mots de **tous** les messages du corpus et affichez la taille du vocabulaire (nombre de colonnes).

In [ ]:
cv_tout = CountVectorizer(tokenizer=nettoyer, lowercase=False, token_pattern=None)
X_tout = cv_tout.fit_transform(df["message"])
print("Taille de la matrice :", X_tout.shape)

## 6 · TF-IDF : donner plus de poids aux mots qui distinguent
Idée simple : un mot présent **partout** (comme « enfant ») apprend peu de chose ; un mot **rare et ciblé** (comme « vaccin ») est un indice précieux. TF-IDF attribue donc un poids plus fort aux mots caractéristiques.

In [ ]:
tfidf = TfidfVectorizer(tokenizer=nettoyer, lowercase=False, token_pattern=None)
X = tfidf.fit_transform(df["message"])
vocab = tfidf.get_feature_names_out()

for cat in ["sante", "etat_civil", "education"]:
    lignes = (df["categorie"] == cat).values
    poids_moyen = X[lignes].mean(axis=0).A1
    top = poids_moyen.argsort()[::-1][:6]
    print(f"{cat:11s} →", ", ".join(vocab[top]))

Les mots en tête de chaque catégorie sont ceux que le modèle utilisera comme **indices**. Vérifiez qu'ils sont cohérents avec ce que vous attendiez.

## 7 · Apprendre à classer : notre premier classifieur
On sépare les données en deux : **75 % pour apprendre**, **25 % pour tester** (des messages que la machine n'a jamais vus). C'est le seul moyen honnête de mesurer sa qualité.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["categorie"], test_size=0.25, stratify=df["categorie"], random_state=42)

modele_mots = make_pipeline(
    TfidfVectorizer(tokenizer=nettoyer, lowercase=False, token_pattern=None),
    LogisticRegression(max_iter=1000))
modele_mots.fit(X_train, y_train)

predictions = modele_mots.predict(X_test)
print("Messages de test :", len(X_test))
print("Taux de bonnes réponses :", round(accuracy_score(y_test, predictions), 2))

In [ ]:
print(classification_report(y_test, predictions, zero_division=0))

**Comment lire ce rapport (sans formule) :**
- **precision** : quand le modèle annonce « santé », a-t-il raison ? (*évite les fausses alertes*)
- **recall** : sur tous les vrais messages « santé », combien en a-t-il retrouvés ? (*évite les oublis*)
- **support** : nombre de messages de cette catégorie dans le test.

**👉 À vous — Exercice 6.** Faites prédire la catégorie de ces 3 nouveaux messages, puis dites si vous êtes d'accord.

In [ ]:
nouveaux = [
    "Où faire vacciner mon petit frère ?",
    "Je veux refaire ma carte d'identité",
    "Quand ouvrent les inscriptions au lycée ?",
]
modele_mots.predict(nouveaux)

## 8 · Regarder les erreurs : la meilleure façon de progresser

In [ ]:
resultats = pd.DataFrame({"message": X_test, "vrai": y_test, "prédit": predictions})
erreurs = resultats[resultats["vrai"] != resultats["prédit"]]
print(len(erreurs), "erreur(s) sur", len(resultats))
erreurs

In [ ]:
phrases_pieges = [
    "Mon fils a besoin d'un certificat médical pour son inscription à l'école.",
    "Je voudrais un acte de naissance pour l'inscription de ma fille au collège.",
    "L'hôpital refuse de me donner une copie de mon dossier.",
]
for p in phrases_pieges:
    print(modele_mots.predict([p])[0].ljust(11), "←", p)

**💬 Discussion (en binôme, 3 minutes).** Ces messages parlent de **deux sujets à la fois**. Que décide le modèle ? Est-ce « faux » ? Que faudrait-il changer : les données, les catégories, ou le modèle ?

## 9 · Robustesse : fautes de frappe et accents manquants
Dans la vraie vie, on écrit vite, sans accents, avec des fautes — et c'est encore plus vrai pour les langues peu dotées en outils. Testons notre modèle sur des messages **abîmés**.

In [ ]:
def abimer(texte, graine=0):
    rnd = random.Random(graine)
    sans_accents = "".join(c for c in unicodedata.normalize("NFD", texte) if unicodedata.category(c) != "Mn")
    mots = sans_accents.split()
    sortie = []
    for m in mots:
        if len(m) > 4 and rnd.random() < 0.6:       # on supprime une lettre dans environ 6 mots longs sur 10
            i = rnd.randrange(1, len(m) - 1)
            m = m[:i] + m[i + 1:]
        sortie.append(m)
    return " ".join(sortie)

print(abimer("Mon enfant a de la fièvre depuis deux jours, où trouver un médecin ?"))

In [ ]:
# Modèle B : au lieu de mots entiers, on regarde des « morceaux de 3 à 5 lettres »
modele_morceaux = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=True),
    LogisticRegression(max_iter=1000))
modele_morceaux.fit(X_train, y_train)

def score_sur_texte_abime(modele, essais=30):
    # on abîme les messages de test de 30 façons différentes et on fait la moyenne
    return sum(accuracy_score(y_test, modele.predict(X_test.apply(lambda t: abimer(t, g))))
               for g in range(essais)) / essais

for nom, m in [("Mots entiers", modele_mots), ("Morceaux de lettres", modele_morceaux)]:
    propre = accuracy_score(y_test, m.predict(X_test))
    print(f"{nom:20s} texte propre : {propre:.2f}   texte abîmé : {score_sur_texte_abime(m):.2f}")

**À retenir :** sur du texte propre, les mots entiers sont aussi bons, voire meilleurs ; mais dès que le texte est abîmé, ils chutent beaucoup plus que les **morceaux de lettres**, qui restent exploitables. C'est l'idée reprise dans le module suivant avec les *sous-mots* des Transformers.

**👉 À vous — Exercice 7 (défi).** Écrivez un message avec des fautes de votre choix et comparez les deux modèles.

In [ ]:
mon_message = ["ou faire vacsiner mon bebe ?"]
print("Mots entiers        :", modele_mots.predict(mon_message)[0])
print("Morceaux de lettres :", modele_morceaux.predict(mon_message)[0])

## 10 · Bilan et limites
**Ce que vous avez construit :** un pipeline complet — *texte → nettoyage → morceaux → nombres → modèle → décision*.

**Ce qu'il faut garder en tête :**
- Le modèle **n'a pas compris** les phrases : il a repéré des indices statistiques.
- Il ne sait traiter que les **catégories vues** à l'entraînement, et se trompe sur les messages à deux sujets.
- Avec **72 exemples**, le résultat est une démonstration, pas un outil de production.
- Les données personnelles réelles ne doivent **jamais** être déposées dans un outil en ligne sans cadre : ici, tout est fictif.

**Pour aller plus loin (module 2) :** comment des *embeddings* et des *Transformers* captent le **sens** des mots, et non plus seulement leur présence.